# EcoSort AI: build the dataset and train the detector

Run the cells from top to bottom in Google Colab.

Before you start: **Runtime → Change runtime type → T4 GPU**.

## 1. Get the code and libraries

In [ ]:
!git clone https://github.com/TheAnakin01/ecosort-ai
%cd ecosort-ai/ai-model
!pip install -q ultralytics roboflow

## 2. Download the three Roboflow datasets

1. Get your API key: Roboflow → Settings → API Keys. Paste it below.
2. For each dataset, open its page on Roboflow Universe. The address looks like
   `universe.roboflow.com/<workspace>/<project>`. Copy those two parts into the list.
3. `version` is the number on the dataset's **Dataset** tab.

Do not save this notebook to GitHub with your API key in it.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"

# (folder name, workspace, project, version)
DATASETS = [
    ("trash-detection", "WORKSPACE", "PROJECT", 1),   # Trash Detection Dataset
    ("trashnet", "WORKSPACE", "PROJECT", 1),          # TrashNet - Annotated Trash
    ("neat-now", "WORKSPACE", "PROJECT", 1),          # Neat Now Waste Detection
]

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
for folder, workspace, project, version in DATASETS:
    rf.workspace(workspace).project(project).version(version).download("yolov8", location=f"downloads/{folder}")

## 3. Check how the classes will be merged

This prints every class in every dataset and which of our four classes it becomes.
Nothing is copied yet.

Read the output carefully:
- A class marked `DROPPED` is left out. If it should be one of ours, add a line for it in `configs/class_overrides.yaml` and run this cell again.
- Look at **Boxes per class**. If `organic` is marked as too few, the model will be weak on organic waste until you add your own photos.

In [ ]:
!python scripts/prepare_dataset.py downloads/trash-detection downloads/trashnet downloads/neat-now --dry-run

## 4. Build the merged dataset

In [ ]:
!rm -rf datasets/ecosort
!python scripts/prepare_dataset.py downloads/trash-detection downloads/trashnet downloads/neat-now

## 5. Train

About 50 epochs is a reasonable first run. Training stops early if the model stops improving.
The last lines print mAP@50 and mAP@50-95 on the test split. Write them in `docs/training-log.md`.

In [ ]:
!python scripts/train.py --epochs 50 --name ecosort-baseline

## 6. Look at the results

In [ ]:
from IPython.display import Image, display

run = "runs/detect/ecosort-baseline"
display(Image(f"{run}/results.png", width=900))
display(Image(f"{run}/confusion_matrix.png", width=700))
display(Image(f"{run}/val_batch0_pred.jpg", width=900))

## 7. Download the trained model

Put the downloaded `best.pt` in `ai-model/weights/` on the machine that runs the backend.

In [ ]:
from google.colab import files

files.download("runs/detect/ecosort-baseline/weights/best.pt")